In [12]:
import os
import pandas as pd

def procesar_hoja_unificada_excel(ruta_excel):
    # 1. Cargamos el archivo de Excel crudo directamente, sin cabeceras fijas
    df_raw = pd.read_excel(ruta_excel, header=None)
    
    # Buscamos las filas donde arranca la palabra 'enero' para identificar cada año
    indices_enero = df_raw[df_raw[0].astype(str).str.strip().str.lower() == 'enero'].index.tolist()
    
    # Lista para acumular las matrices anuales limpias
    registros_historicos = []
    
    # Mapeamos los años correspondientes a los bloques encontrados secuencialmente
    años = [2023, 2024, 2025, 2026]
    
    for i, idx_enero in enumerate(indices_enero):
        if i >= len(años):
            break
        año_actual = años[i]
        
        # Cada bloque mensual dura exactamente 12 filas (enero a diciembre)
        df_bloque = df_raw.iloc[idx_enero:idx_enero+12, :].copy()
        
        # Estructuramos la data limpia del bloque anual
        df_año = pd.DataFrame()
        
        # Mapeo de fechas (Año-Mes-01) para series temporales
        meses_map = {
            'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
            'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
        }
        
        df_bloque[0] = df_bloque[0].astype(str).str.strip().str.lower()
        df_año['Mes_Num'] = df_bloque[0].map(meses_map)
        df_año['Año'] = año_actual
        
        # Creamos la columna Fecha con formato datetime real
        df_año['Fecha'] = pd.to_datetime(df_año['Año'].astype(str) + '-' + df_año['Mes_Num'].astype(str) + '-01')
        
        # =========================================================================
        # ACÁ METIMOS EL CÓDIGO NUEVO: Extracción con selección dinámica de columnas
        # =========================================================================
        df_año['kWh_Medidor_1'] = pd.to_numeric(df_bloque[1], errors='coerce')
        df_año['Produccion_lts_kg'] = pd.to_numeric(df_bloque[6], errors='coerce')

        if año_actual == 2023:
            df_año['kWh_Medidor_2'] = 0.0  # No estaba conectado
        elif año_actual == 2024:
            df_año['kWh_Medidor_2'] = pd.to_numeric(df_bloque[10], errors='coerce')
        elif año_actual in [2025, 2026]:
            df_año['kWh_Medidor_2'] = pd.to_numeric(df_bloque[11], errors='coerce')
        # =========================================================================
            
        # Filtramos solo las columnas necesarias para el modelo predictivo
        df_año = df_año[['Fecha', 'kWh_Medidor_1', 'kWh_Medidor_2', 'Produccion_lts_kg']]
        
        registros_historicos.append(df_año)
        
    # 2. Concatenamos verticalmente todos los años en una única línea de tiempo
    df_consolidado = pd.concat(registros_historicos, ignore_index=True)
    
    # 3. Eliminamos filas del futuro (meses de 2026 que todavía no se ejecutaron)
    df_consolidado = df_consolidado.dropna(subset=['kWh_Medidor_1', 'Produccion_lts_kg'], how='all')
    
    # Ordenamos cronológicamente
    df_consolidado = df_consolidado.sort_values('Fecha').reset_index(drop=True)
    
    return df_consolidado


# --- EJECUCIÓN DEL PIPELINE ---

# Apuntamos al archivo .xlsx en la carpeta correspondiente
ruta_entrada = 'data/raw/Medidores copilados.xlsx'

# Ejecutamos la función modificada
df_final = procesar_hoja_unificada_excel(ruta_entrada)

# Creamos la carpeta processed si no existe
os.makedirs('data/processed', exist_ok=True)

# Guardamos la matriz corregida y sin NaNs lista para usar
df_final.to_csv('data/processed/features_energia_tecnomyl.csv', index=False)

print("=========================================================")
print("¡DATASET HISTÓRICO CONSOLIDADO SIN ERRORES EN MEDIDOR 2!")
print("=========================================================")
print(df_final.to_string(index=False))

¡DATASET HISTÓRICO CONSOLIDADO SIN ERRORES EN MEDIDOR 2!
     Fecha  kWh_Medidor_1  kWh_Medidor_2  Produccion_lts_kg
2023-01-01      264947.00            0.0            1114850
2023-02-01      517814.00            0.0            2329571
2023-03-01      428057.00            0.0            1751127
2023-04-01      420763.00            0.0            2720075
2023-05-01      406760.00            0.0            1147140
2023-06-01      396362.00            0.0            2441754
2023-07-01      500485.00            0.0            2751160
2023-08-01      479515.00            0.0            3263219
2023-09-01      430806.00            0.0            2874751
2023-10-01      350585.00            0.0            2692960
2023-11-01      309975.00            0.0            1379860
2023-12-01      109030.00            0.0            1156484
2024-01-01      296735.00         1373.0            2196843
2024-02-01      347635.00         1352.0            2969960
2024-03-01      319955.00         1863.0   

In [1]:
import os
import numpy as np
import pandas as pd

def procesar_hoja_unificada_excel(ruta_excel, ruta_gas):
    df_raw = pd.read_excel(ruta_excel, header=None)
    
    indices_enero = df_raw[df_raw[0].astype(str).str.strip().str.lower() == 'enero'].index.tolist()
    
    registros_historicos = []
    años = [2023, 2024, 2025, 2026]
    
    meses_map = {
        'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6,
        'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
    }
    
    for i, idx_enero in enumerate(indices_enero):
        if i >= len(años):
            break
        año_actual = años[i]
        
        # Cada bloque mensual dura exactamente 12 filas (enero a diciembre)
        df_bloque = df_raw.iloc[idx_enero:idx_enero+12, :].copy()
        
        df_año = pd.DataFrame()
        
        df_bloque[0] = df_bloque[0].astype(str).str.strip().str.lower()
        
        df_año['Mes_Num'] = df_bloque[0].map(meses_map)
        df_año['Año'] = año_actual
        df_año['Fecha'] = pd.to_datetime(df_año['Año'].astype(str) + '-' + df_año['Mes_Num'].astype(str) + '-01')
        df_año['kWh_Medidor_1'] = pd.to_numeric(df_bloque[1], errors='coerce')
        df_año['Produccion_lts_kg'] = pd.to_numeric(df_bloque[6], errors='coerce')

        if año_actual == 2023:
            df_año['kWh_Medidor_2'] = 0.0
        elif año_actual == 2024:
            df_año['kWh_Medidor_2'] = pd.to_numeric(df_bloque[10], errors='coerce')
        elif año_actual in [2025, 2026]:
            df_año['kWh_Medidor_2'] = pd.to_numeric(df_bloque[11], errors='coerce')
        
        # Filtramos solo las columnas necesarias para el modelo predictivo
        df_año = df_año[['Fecha', 'kWh_Medidor_1', 'kWh_Medidor_2', 'Produccion_lts_kg']]
        
        registros_historicos.append(df_año)
        
    # Concatenamos verticalmente todos los años en una única línea de tiempo
    df_consolidado = pd.concat(registros_historicos, ignore_index=True)
    
    # Eliminamos filas del futuro (meses de 2026 que todavía no se ejecutaron)
    df_consolidado = df_consolidado.dropna(subset=['kWh_Medidor_1', 'Produccion_lts_kg'], how='all')
    
    # PROCESAMIENTO E INTEGRACIÓN DE GAS NATURAL
    df_raw_gas = pd.read_excel(ruta_gas, header=None)
    indices_enero_gas = df_raw_gas[df_raw_gas[0].astype(str).str.strip().str.lower() == 'enero'].index.tolist()
    gas_historico = []
    
    for i, idx_enero in enumerate(indices_enero_gas):
        if i >= len(años):
            break
        año_actual = años[i]
        df_bloque_gas = df_raw_gas.iloc[idx_enero:idx_enero+12, :].copy()
        
        df_año_gas = pd.DataFrame()
        df_bloque_gas[0] = df_bloque_gas[0].astype(str).str.strip().str.lower()
        df_año_gas['Mes_Num'] = df_bloque_gas[0].map(meses_map)
        df_año_gas['Año'] = año_actual
        df_año_gas['Fecha'] = pd.to_datetime(df_año_gas['Año'].astype(str) + '-' + df_año_gas['Mes_Num'].astype(str) + '-01')
        df_año_gas['Consumo_Gas_m3'] = pd.to_numeric(df_bloque_gas[1], errors='coerce')
        
        gas_historico.append(df_año_gas[['Fecha', 'Consumo_Gas_m3']])
        
    df_gas = pd.concat(gas_historico, ignore_index=True)
    df_gas = df_gas.dropna(subset=['Fecha']).reset_index(drop=True)
    
    # Cruce horizontal con la tabla de energía
    df_consolidado = pd.merge(df_consolidado, df_gas, on='Fecha', how='left')
    
    # Interpolación de valores faltantes de gas (meses vacíos de 2024 y 2026)
    df_consolidado['Consumo_Gas_m3'] = df_consolidado['Consumo_Gas_m3'].interpolate(method='linear').ffill().bfill()
    
    # Variable de cambio estructural por encendido industrial del Medidor 2
    df_consolidado['Sopladoras_Activas'] = np.where(df_consolidado['Fecha'].dt.year >= 2025, 1, 0)
    # =========================================================================

    # Ordenamos cronológicamente
    df_consolidado = df_consolidado.sort_values('Fecha').reset_index(drop=True)
    
    return df_consolidado


# --- EJECUCIÓN DEL PIPELINE ---

# Apuntamos a los archivos correspondientes en la carpeta cruda
ruta_entrada = 'data/raw/Medidores copilados.xlsx'
ruta_gas = 'data/raw/Resumen gas.xlsx'

# Ejecutamos la función modificada pasándole ambos archivos
df_final = procesar_hoja_unificada_excel(ruta_entrada, ruta_gas)

# Creamos la carpeta processed si no existe
os.makedirs('data/processed', exist_ok=True)

# Guardamos la matriz corregida y sin NaNs lista para usar
df_final.to_csv('data/processed/features_energia_tecnomyl.csv', index=False)

print("=========================================================")
print("¡DATASET HISTÓRICO CONSOLIDADO CON GAS REAL CON EXITO!")
print("=========================================================")
print(df_final.to_string(index=False))

¡DATASET HISTÓRICO CONSOLIDADO CON GAS REAL CON EXITO!
     Fecha  kWh_Medidor_1  kWh_Medidor_2  Produccion_lts_kg  Consumo_Gas_m3  Sopladoras_Activas
2023-01-01      264947.00            0.0            1114850        43.38383                   0
2023-02-01      517814.00            0.0            2329571        74.46603                   0
2023-03-01      428057.00            0.0            1751127       101.79547                   0
2023-04-01      420763.00            0.0            2720075        80.06237                   0
2023-05-01      406760.00            0.0            1147140        68.74683                   0
2023-06-01      396362.00            0.0            2441754        37.76531                   0
2023-07-01      500485.00            0.0            2751160       127.90123                   0
2023-08-01      479515.00            0.0            3263219       131.69097                   0
2023-09-01      430806.00            0.0            2874751       115.90814      